## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

**Caution**... This notebook requires significant execution time as there are 9,319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer with a typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and the use of cloud computing services.


## Landsat Data Extraction (Phase 2)

This updated process focuses on improving spectral signal quality by narrowing the spatial focus and adding parameters critical for water quality analysis.

**Key Improvements:**
* **Refined Buffer (~45m):** Reduced from 100m to 45m (`bbox_size = 0.0004`) to minimize terrestrial vegetation interference.
* **Enhanced Band Selection:** Now includes Blue and Red bands for turbidity and chlorophyll proxy calculations.
* **On-the-fly Indices:** Automatically calculates **NDVI** (vegetation detection) and **Turbidity** (Red/Blue ratio) during extraction.
* **Batch Processing & Resume Logic:** Includes a checkpoint system to handle API timeouts and long execution times (~7 hours for the full dataset).

### Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process. 

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

import os
from tqdm import tqdm
tqdm.pandas()

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

### Extracting Landsat Data Using API Calls (Updated for Phase 2)

The API-based method allows us to efficiently access Landsat data for specific coordinates and time periods, ensuring scalability and reproducibility of the process.

Through the API, we can query individual bands or compute indices on the fly, reducing storage requirements and simplifying data preprocessing. This updated version of the extraction process focuses on improving the spectral signal by narrowing the spatial focus and adding parameters critical for water quality analysis.

The compute_Landsat_values function extracts Landsat surface reflectance values for specific sampling locations using a refined 45 m focal buffer around each point. For each location:

* A bounding box (bbox) is created around the latitude and longitude coordinates.

* The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.

* The nearest low-cloud (<10% cloud cover) scene is selected.

* Expanded Band Selection: In addition to the original bands, we now load Blue and Red bands to enable the calculation of turbidity and chlorophyll-related proxies.

* Median Aggregation: Median values of the pixels within the bounding box are computed to reduce the effect of noise, outliers, or edge effects.

* On-the-fly Feature Engineering: The function now automatically calculates NDVI (to detect vegetation interference) and Turbidity (Red/Blue ratio) to provide more descriptive power to the machine learning model.

**Why the buffer value is 0.0004**

In Phase 1, a ~100 m buffer (0.00089831) was used. However, to minimize "spectral bleeding" from riverbanks and terrestrial vegetation—which can skew water quality readings—we have reduced the buffer to ~45 m.

At the equator, 1 degree ≈ 110,000 m.
The degree equivalent for our refined buffer is:

buffer_deg ≈ 45 m / 110,000 m per degree ≈ 0.0004

This ensures the extraction captures pixels primarily from the water body, providing a "cleaner" spectral signature for the training of our predictive models.


In [2]:
# Setup
tqdm.pandas()

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    # Manejo de fecha robusto
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # IMPROVEMENT: Reduced buffer (~45m) to avoid capturing land edges
    # 0.00089831 was for 100m. We will use 0.0004 for an area closer to clear water.
    bbox_size = 0.0004  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # IMPROVEMENT: Stricter cloud filter if possible
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "blue": np.nan, "red": np.nan,
            "swir16": np.nan, "swir22": np.nan, "NDVI": np.nan, "Turbidity": np.nan
        })

    try:
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # IMPROVEMENT: We've added 'red' and 'blue' for new indices
        bands_of_interest = ["green", "nir08", "swir16", "swir22", "red", "blue"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        # Convert to float and handle nulls
        bands_data = {}
        for b in bands_of_interest:
            val = float(data[b].astype("float").median(skipna=True).values)
            bands_data[b] = val if val != 0 else np.nan

        # FEATURE ENGINEERING: Calculation of indices within the extraction
        eps = 1e-10
        # NDVI: Useful for determining if algae are present or if the buffer has touched land
        ndvi = (bands_data["nir08"] - bands_data["red"]) / (bands_data["nir08"] + bands_data["red"] + eps)
        # Turbidity: Basic turbidity (Red/Blue ratio is a common proxy)
        turbidity = bands_data["red"] / (bands_data["blue"] + eps)
        
        return pd.Series({
            "nir": bands_data["nir08"],
            "green": bands_data["green"],
            "blue": bands_data["blue"],
            "red": bands_data["red"],
            "swir16": bands_data["swir16"],
            "swir22": bands_data["swir22"],
            "NDVI": ndvi,
            "Turbidity": turbidity
        })
    
    except Exception:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "blue": np.nan, "red": np.nan,
            "swir16": np.nan, "swir22": np.nan, "NDVI": np.nan, "Turbidity": np.nan
        })

### Extracting features for the training dataset

In [3]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
display(Water_Quality_df.head())

In [4]:
Water_Quality_df.shape

In [5]:
Water_Quality_df_200 = Water_Quality_df.loc[0:199]
Water_Quality_df_200.shape

In [ ]:
# Create the ID based on the index so you can perform the merge later
Water_Quality_df = Water_Quality_df.reset_index().rename(columns={'index': 'dis_detail_id'})

print(Water_Quality_df.columns)

### Note

The Landsat data extraction process for all 9,319 locations typically requires more than 7 hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.

In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.

We have already executed the full extraction for all 9,319 locations and saved the output to **landsat_features_training.csv**, which will be used in the benchmark notebook.  
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as **landsat_features_training.csv** to ensure the benchmark notebook runs smoothly.


In [ ]:
import os

# 1. Define the path of the new improved file
train_features_path = "landsat_features_fase2_PRO.csv"

# 2. Batch configuration
batch_size = 500 
total_records = len(Water_Quality_df) # Use the complete dataframe, not just the 200 records

print(f"🚀 Starting extraction Phase 2 for {total_records} records...")

for i in range(0, total_records, batch_size):
    # Select the current batch
    batch_df = Water_Quality_df.iloc[i:i+batch_size].copy()
    
    print(f"📦 Processing batch: {i} to {min(i+batch_size, total_records)}...")
    
    landsat_results = batch_df.progress_apply(compute_Landsat_values, axis=1)
    
    # Combine with an ID column (dis_detail_id) to ensure a successful merge
    batch_final = pd.concat([batch_df[['dis_detail_id']], landsat_results], axis=1)
    
    # Save or Append to CSV file
    if not os.path.isfile(train_features_path):
        batch_final.to_csv(train_features_path, index=False)
    else:
        # mode='a' adds to the end of the file; header=False prevents repeating column headings
        batch_final.to_csv(train_features_path, mode='a', header=False, index=False)
        
    print(f"✅ Batch saved successfully in {train_features_path}")

print("✨ ✅ Extraction complete! You now have your Phase 2 variables safe.")

In [ ]:
train_features_path = "landsat_features_fase2_PRO.csv"

# 1. See how many rows we've already saved
if os.path.exists(train_features_path):
    start_index = len(pd.read_csv(train_features_path))
    print(f"✅ We rescued {start_index} rows. Continuing from there...")
else:
    start_index = 0

# 2. Improved extraction loop
batch_size = 500
for i in range(start_index, len(Water_Quality_df), batch_size):
    batch_df = Water_Quality_df.iloc[i:i+batch_size].copy()
    print(f"📦 Processing batch: {i} to {min(i+batch_size, len(Water_Quality_df))}...")
    
    try:
        landsat_results = batch_df.progress_apply(compute_Landsat_values, axis=1)
        
        # Join and Save
        batch_final = pd.concat([batch_df[['dis_detail_id']], landsat_results], axis=1)
        batch_final.to_csv(train_features_path, mode='a', header=(not os.path.exists(train_features_path)), index=False)
        
    except Exception as e:
        print(f"⚠️ Connection error. The server is overloaded. Please try again in a moment.")
        break 

**NDMI and MNDWI Indices**

In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:

- **NDMI (Normalized Difference Moisture Index):**  
  Measures vegetation water content and surface moisture.  
  Computed as *(NIR - SWIR16) / (NIR + SWIR16)*.

- **MNDWI (Modified Normalized Difference Water Index):**  
  Highlights open water features by enhancing water reflectance and suppressing built-up areas.  
  Computed as *(Green - SWIR16) / (Green + SWIR16)*.

An **epsilon value** (*eps = 1e-10*) is added to the denominators to avoid division by zero.  
These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.


In [ ]:
# 1. Read the consolidated file that we created in batches
train_features_path = "landsat_features_fase2_PRO.csv"
landsat_train_features = pd.read_csv(train_features_path)

In [ ]:
# 2. Join with the original coordinates if you did not include them in the batch

landsat_train_features = landsat_train_features.merge(
    Water_Quality_df[['dis_detail_id', 'Latitude', 'Longitude', 'Sample Date']], 
    on='dis_detail_id', 
    how='left'
)

In [ ]:
# 3. Reorder columns
# Verify which columns actually exist in the file
print("Current columns:", landsat_train_features.columns.tolist())

# Define epsilon to avoid division by zero
eps = 1e-10

# Create NDWI/NDMI if they don't exist (We ensure compatibility)
# NDMI uses NIR and SWIR16
if 'nir' in landsat_train_features.columns and 'swir16' in landsat_train_features.columns:
    landsat_train_features['NDWI'] = (landsat_train_features['green'] - landsat_train_features['nir']) / (landsat_train_features['green'] + landsat_train_features['nir'] + eps)
    landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)
   
    landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)

# 4. Adjust the column list 
cols_disponibles = [c for c in ['dis_detail_id', 'Latitude', 'Longitude', 'Sample Date', 
                               'nir', 'green', 'swir16', 'swir22', 'red', 'blue', 
                               'NDVI', 'NDWI', 'MNDWI', 'NDMI', 'Turbidity'] 
                    if c in landsat_train_features.columns]

landsat_train_features = landsat_train_features[cols_disponibles]

print("✅ Columns successfully reordered.")
display(landsat_train_features.head())

In [ ]:
# 4. Save and upload to Snowflake
landsat_train_features.to_csv("/tmp/landsat_features_training_fase2.csv", index=False)

session.sql("""
    PUT file:///tmp/landsat_features_training_fase2.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("✅ Training Features (Phase 2) saved and uploaded!")

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.

### Extracting features for the validation dataset

In [ ]:
# --- EXTRACTION FOR VALIDATION---
Validation_df = pd.read_csv('submission_template.csv')

# 1. Automatically detect the ID column name
posibles_nombres_id = ['dis_detail_id', 'ID', 'id', 'index']
columna_id = next((c for c in posibles_nombres_id if c in Validation_df.columns), None)

# 2. If no ID column is found, create one based on the index
if columna_id is None:
    print("⚠️ No ID column detected, creating one based on the index...")
    Validation_df = Validation_df.reset_index().rename(columns={'index': 'dis_detail_id'})
    columna_id = 'dis_detail_id'
else:
    print(f"✅ Using '{columna_id}' as unique identifier.")

val_output_path = "landsat_features_validation_fase2.csv"
batch_size = 500

print(f"🚀 Starting extraction for VALIDATION (Phase 2) - {len(Validation_df)} records...")

for i in range(0, len(Validation_df), batch_size):
    batch = Validation_df.iloc[i:i+batch_size].copy()
    print(f"📦 Validation Batch: {i} a {min(i+batch_size, len(Validation_df))}")
    
    # Extract using the enhanced function
    results = batch.progress_apply(compute_Landsat_values, axis=1)
    
    # Combine using the detected column + coordinates
    cols_a_mantener = [c for c in [columna_id, 'Latitude', 'Longitude', 'Sample Date'] if c in batch.columns]
    batch_final = pd.concat([batch[cols_a_mantener], results], axis=1)
    
    # Save
    if not os.path.isfile(val_output_path):
        batch_final.to_csv(val_output_path, index=False)
    else:
        batch_final.to_csv(val_output_path, mode='a', header=False, index=False)

print("✨ Validation extraction completed.")

In [ ]:
# 1. Ensure that the route is absolute
full_val_path = os.path.abspath(val_output_path)

# 2. Upload to Snowflake using the universal route @~/ (User Stage)
try:
    session.sql(f"""
        PUT file://{full_val_path} 
        @~/versions/live/
        AUTO_COMPRESS=FALSE
        OVERWRITE=TRUE
    """).collect()
    print("✅ Validation file successfully uploaded to personal Stage!")
except Exception as e:
    print(f"❌ Error uploading: {e}")
    print("Attempting an alternative route...")
    # Attempt 2: No subfolders just in case
    session.sql(f"""
        PUT file://{full_val_path} 
        @~/
        AUTO_COMPRESS=FALSE
        OVERWRITE=TRUE
    """).collect()

print("✨ The file is now visible in Snowflake. Refresh the left panel.")

In [ ]:

# SAVE TRAINING 
train_file = "landsat_features_training_fase2_final.csv"
landsat_train_features.to_csv(train_file, index=False)
print(f"✅ Training file saved locally as: {os.path.abspath(train_file)}")

# SAVE VALIDATION 
val_file = "landsat_features_validation_fase2_final.csv"
landsat_val_features.to_csv(val_file, index=False)
print(f"✅ Validation file saved locally as: {os.path.abspath(val_file)}")

print("\n💡 INSTRUCTION: Go to the left panel, find these files, right-click them, and select 'Download'.")